# Monitoring & Drift Detection — AI Lab Instructor Notebook

*MLOps & Systems · Expert*



5 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)


## Task 1: Dataset & Setup

# Monitoring & Drift Detection — Student Lab

## Task 0.0 — Setup

Load NumPy and the small `check(...)` helper used across the notebook. All later sections build on this setup.

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)


**Why this works.** # Monitoring & Drift Detection — Instructor Lab

Offline lab: detect feature drift and performance drift across simulated production batches.

## 0 — Generate baseline + drifting production batches

## Task 2: 0 — Generate baseline + drifting production batches

## Section 0 — Generate baseline + drifting production batches

### Task 0.1
Implement the synthetic batch generator used throughout the lab.

- `sigmoid(z)`: logistic link
- `make_batch(...)`: create one batch of features and labels
- inject drift into the first `k = d//4` features using `mean_shift` and `scale_shift`
- return `X` with shape `(n, d)` and binary labels `y` with shape `(n,)`

Then create one reference batch `X0, y0` and six production batches in `BATCHES`.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def make_batch(n: int, d: int, seed: int, mean_shift: float = 0.0, scale_shift: float = 0.0, label_shift: float = 0.0):
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n, d)).astype(np.float64)
    k = max(1, d // 4)
    X[:, :k] = (1.0 + scale_shift) * X[:, :k] + mean_shift
    w_true = np.linspace(-1.0, 1.0, d)
    b_true = 0.0
    p = sigmoid(X @ w_true + b_true)
    p = np.clip(p + label_shift, 0.0, 1.0)
    y = (p >= 0.5).astype(np.int64)
    return X, y

N, D = 5000, 12
X0, y0 = make_batch(N, D, seed=0)
BATCHES = []
for t in range(6):
    Xb, yb = make_batch(N, D, seed=10+t, mean_shift=0.05*t, scale_shift=0.03*t)
    BATCHES.append((Xb, yb))

check('shapes', X0.shape == (N, D) and y0.shape == (N,))


In [ ]:
# Checks
check('shapes', X0.shape == (N, D) and y0.shape == (N,))

## Task 3: 1 — Drift metrics (PSI + KS + mean/variance shift)

## Section 1 — Drift metrics (PSI + KS + mean/variance shift)

### Task 1.1
Implement feature-level drift metrics between a reference batch and a production batch.

- `psi_1d(base, prod)`: quantile-binned PSI for one feature
- `ks_statistic(base, prod)`: two-sample KS statistic without SciPy
- `mean_var_shift(base, prod)`: return mean shift, absolute mean shift, and variance ratio
- `drift_report(X_base, X_prod)`: compute one record per feature and sort by strongest drift

Print the top three most drifted features for the first production batch.

In [ ]:
$f

In [ ]:
# Checks
check('rep_len', len(rep0) == D)

## Task 4: 2 — Model + performance monitoring

## Section 2 — Model + performance monitoring

### Task 2.1
Build a deterministic batch-level monitoring loop for model performance.

- `fit_fixed_model(d)`: create one fixed linear model
- `predict_proba(model, X)`: convert logits to probabilities
- `accuracy(y, yhat)`: classification accuracy
- `log_loss(y, p)`: binary cross-entropy
- `eval_batch(model, X, y)`: return a metrics dictionary with `acc` and `log_loss`

Then evaluate the reference batch and every production batch and store the metrics in `series`.

In [ ]:
def fit_fixed_model(d: int):
    w = np.linspace(-0.8, 0.8, d)
    b = 0.0
    return {"w": w, "b": b}

def predict_proba(model, X):
    return sigmoid(X @ model["w"] + model["b"])

def accuracy(y, yhat):
    return float(np.mean(y == yhat))

def log_loss(y, p, eps: float = 1e-12):
    p = np.clip(p, eps, 1.0 - eps)
    return float(-np.mean(y*np.log(p) + (1-y)*np.log(1-p)))

def eval_batch(model, X, y):
    p = predict_proba(model, X)
    yhat = (p >= 0.5).astype(np.int64)
    return {"acc": accuracy(y, yhat), "log_loss": log_loss(y, p)}

model = fit_fixed_model(D)
base_metrics = eval_batch(model, X0, y0)
print('base', base_metrics)

series = []
for t, (Xb, yb) in enumerate(BATCHES):
    m = eval_batch(model, Xb, yb)
    series.append(m)
print('series', series)


In [ ]:
# Checks
check('series_len', len(series) == len(BATCHES))
check('metrics_finite', all(np.isfinite(m['acc']) and np.isfinite(m['log_loss']) for m in [base_metrics] + series))
check('acc_unit_interval', all(0.0 <= m['acc'] <= 1.0 for m in [base_metrics] + series))


## Task 5: 3 — Alerting rules (drift + performance)

## Section 3 — Alerting rules (drift + performance)

### Task 3.1
Combine input-drift monitoring and outcome monitoring into one alert object.

- compute the ranked drift report for the current batch
- use the top feature to decide whether drift exceeds `psi_thresh` or `ks_thresh`
- trigger `perf_alert` when accuracy drops enough relative to baseline
- return a compact alert dictionary for logging or paging

Finally compare an early batch with a late batch and confirm that the later batch shows stronger PSI drift.

In [ ]:
$10

In [ ]:
# Checks
check('psi_increases', a5['top_psi'] >= a0['top_psi'])